# MCTNet — Entraînement Final (Arkansas & California)

**Pipeline :**
1. Montage Drive + dépendances
2. Configuration centralisée
3. Préparation des datasets `.npz / .json`
4. Validation des dimensions
5. Entraînement sur Arkansas & California
6. Visualisation : courbes d'apprentissage + matrices de confusion
7. Tableau récapitulatif des métriques


---
## Cellule 1 — Montage Google Drive & dépendances

In [ ]:
# ── Montage Drive ───────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive monté.')
except ImportError:
    IN_COLAB = False
    print('Exécution locale détectée.')

# ── Dépendances ─────────────────────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Torch est déjà présent sur Colab ; on s'assure juste du reste.
pip('numpy', 'pandas', 'matplotlib', 'scikit-learn')

import json, shutil
from pathlib import Path
from types  import SimpleNamespace

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f'PyTorch {torch.__version__} | GPU : {torch.cuda.is_available()}')

---
## Cellule 2 — Configuration centralisée

> **Modifiez uniquement cette cellule** pour changer d'état, de nombre d'époques, etc.

In [ ]:
# ── Dossiers ─────────────────────────────────────────────────────────────────
# Dossier contenant vos scripts Python (build_dataset.py, mctnet_model.py, etc.)
PROJECT_DIR   = Path('/content/drive/MyDrive/New project/python')

# Dossier racine des exports GEE
BASE_DIR      = Path('/content/drive/MyDrive/mctnet_crop_mapping_2021')

# Fichiers CSV bruts exportés depuis Google Earth Engine
CSV_AR = BASE_DIR / 'mctnet_samples_AR_2021.csv'
CSV_CA = BASE_DIR / 'mctnet_samples_CA_2021.csv'

# Dossier des datasets prétraités
PROCESSED_DIR = BASE_DIR / 'processed'

# Dossier des runs d'entraînement
RUNS_DIR = BASE_DIR / 'training_runs'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparamètres ──────────────────────────────────────────────────────────
CFG = SimpleNamespace(
    epochs                  = 100,
    batch_size              = 32,
    learning_rate           = 1e-3,
    weight_decay            = 1e-4,
    dropout                 = 0.1,
    n_stages                = 3,
    n_heads                 = 5,
    kernel_size             = 3,
    seed                    = 2021,
    num_workers             = 0,
    early_stopping_patience = 15,
    cpu                     = False,
    no_alpe                 = False,
    no_mask                 = False,
    no_cnn                  = False,
    no_trans                = False,
    # États à traiter
    states                  = ['arkansas', 'california'],
)

# Ajouter PROJECT_DIR au path Python pour les imports
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Configuration OK.')
print(f'  CSV AR  : {CSV_AR}  (existe = {CSV_AR.exists()})')
print(f'  CSV CA  : {CSV_CA}  (existe = {CSV_CA.exists()})')
print(f'  Scripts : {PROJECT_DIR}  (existe = {PROJECT_DIR.exists()})')

---
## Cellule 3 — Préparation des datasets `.npz / .json`

Lance `build_dataset.py` pour convertir les CSV GEE en tenseurs PyTorch.

In [ ]:
from build_dataset import build_dataset_bundle, save_bundle

for csv_path in [CSV_AR, CSV_CA]:
    if not csv_path.exists():
        raise FileNotFoundError(
            f'CSV introuvable : {csv_path}\n'
            f'Vérifiez que le fichier est bien dans {BASE_DIR}'
        )

    bundle, meta = build_dataset_bundle(
        csv_path             = csv_path,
        normalize_reflectance= True,
        reflectance_scale    = 10000.0,
        split_seed           = CFG.seed,
    )

    slug      = meta['state_name'].lower().replace(' ', '_')
    npz_path  = PROCESSED_DIR / f'{slug}_mctnet_dataset.npz'
    json_path = PROCESSED_DIR / f'{slug}_mctnet_dataset.json'
    save_bundle(bundle, meta, npz_path, json_path)

    # Résumé compact
    print(f'\n[{meta["state_name"]}]')
    print(f'  {meta["num_classes"]} classes : {list(meta["class_name_to_index"].keys())}')
    for split, counts in meta['split_counts'].items():
        n = meta['samples_per_split'][split]
        print(f'  {split:5s} ({n:5d} éch.) : {counts}')
    print(f'  Sauvegardé → {npz_path.name}')

print('\nPréparation terminée.')

---
## Cellule 4 — Validation des dimensions des tenseurs

In [ ]:
from validate_mctnet_tensors import load_bundle, validate_split_shapes, validate_dataloader

all_ok = True
for npz_path in sorted(PROCESSED_DIR.glob('*_mctnet_dataset.npz')):
    slug      = npz_path.stem.replace('_mctnet_dataset', '')
    json_path = PROCESSED_DIR / f'{slug}_mctnet_dataset.json'
    bundle    = load_bundle(npz_path)
    meta      = json.loads(json_path.read_text(encoding='utf-8'))

    print(f'\n{"-"*60}')
    print(f'État : {meta["state_name"]}  |  Classes : {meta["class_name_to_index"]}')
    print(f'Formes x_train : {bundle["x_train"].shape}  (attendu : [N, 36, 10])')

    try:
        for split in ['train', 'val', 'test']:
            validate_split_shapes(bundle, split)
            validate_dataloader(bundle, split, batch_size=CFG.batch_size)
    except AssertionError as e:
        print(f'  ERREUR : {e}')
        all_ok = False

if all_ok:
    print('\n✓ Toutes les dimensions sont correctes. Prêt pour l\'entraînement.')
else:
    print('\n✗ Des erreurs de dimensions ont été détectées. Vérifiez build_dataset.py.')

---
## Cellule 5 — Entraînement MCTNet sur Arkansas & California

Cette cellule appelle `run_mctnet_training.train_model()` pour chaque état.
Les checkpoints, métriques et courbes sont sauvegardés dans `RUNS_DIR`.

In [ ]:
from run_mctnet_training import (
    set_seed, load_bundle, build_dataloaders,
    train_model, confusion_matrix_from_predictions,
    classification_metrics, plot_confusion_matrix,
)

set_seed(CFG.seed)
all_summaries = {}

for state_slug in CFG.states:
    npz_path  = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.npz'
    json_path = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.json'

    if not npz_path.exists():
        print(f'Dataset manquant pour {state_slug}, ignoré.')
        continue

    bundle   = load_bundle(npz_path)
    metadata = json.loads(json_path.read_text(encoding='utf-8'))
    state_run_dir = RUNS_DIR / state_slug
    state_run_dir.mkdir(parents=True, exist_ok=True)

    # Construction des args pour train_model
    train_args = SimpleNamespace(
        dataset_npz             = str(npz_path),
        metadata_json           = str(json_path),
        output_dir              = str(state_run_dir),
        checkpoint_path         = str(state_run_dir / 'best_mctnet.pt'),
        epochs                  = CFG.epochs,
        batch_size              = CFG.batch_size,
        learning_rate           = CFG.learning_rate,
        weight_decay            = CFG.weight_decay,
        dropout                 = CFG.dropout,
        n_stages                = CFG.n_stages,
        n_heads                 = CFG.n_heads,
        kernel_size             = CFG.kernel_size,
        seed                    = CFG.seed,
        num_workers             = CFG.num_workers,
        early_stopping_patience = CFG.early_stopping_patience,
        cpu                     = CFG.cpu,
        no_alpe                 = CFG.no_alpe,
        no_mask                 = CFG.no_mask,
        no_cnn                  = CFG.no_cnn,
        no_trans                = CFG.no_trans,
    )

    model, metrics_payload = train_model(
        args=train_args, bundle=bundle, metadata=metadata
    )

    # Sauvegarde des métriques
    (state_run_dir / 'metrics.json').write_text(
        json.dumps(metrics_payload, indent=2), encoding='utf-8'
    )

    # Matrice de confusion depuis le meilleur checkpoint
    from mctnet_model import MCTNet, MCTNetConfig
    device = torch.device('cuda' if torch.cuda.is_available() and not CFG.cpu else 'cpu')
    ckpt   = torch.load(train_args.checkpoint_path, map_location=device)
    best_m = MCTNet(MCTNetConfig(**ckpt['model_config'])).to(device)
    best_m.load_state_dict(ckpt['model_state_dict'])
    best_m.eval()

    loaders  = build_dataloaders(bundle, CFG.batch_size, CFG.num_workers)
    yt_l, yp_l = [], []
    with torch.no_grad():
        for batch in loaders['test']:
            logits = best_m(batch['x'].to(device), batch['valid_mask'].to(device))
            yt_l.append(batch['y'].numpy())
            yp_l.append(logits.argmax(1).cpu().numpy())

    y_true = np.concatenate(yt_l)
    y_pred = np.concatenate(yp_l)
    n_cls  = len(metadata['class_name_to_index'])
    cm     = confusion_matrix_from_predictions(y_true, y_pred, n_cls)
    c_names = [None] * n_cls
    for name, idx in metadata['class_name_to_index'].items():
        c_names[int(idx)] = name

    plot_confusion_matrix(
        cm, c_names,
        state_run_dir / 'test_confusion_matrix.png',
        state_name=metadata['state_name'],
    )

    all_summaries[state_slug] = {
        'state_name':       metadata['state_name'],
        'num_classes':      n_cls,
        'best_val':         metrics_payload['best_val'],
        'test_at_best_val': metrics_payload['test_at_best_val'],
        'class_names':      c_names,
        'confusion_matrix': cm.tolist(),
        'history':          metrics_payload['history'],
    }
    print(f'\n✓ [{metadata["state_name"]}] Entraînement terminé.')
    print(f'  Meilleure val  : {metrics_payload["best_val"]}')
    print(f'  Test           : {metrics_payload["test_at_best_val"]}')

print('\nEntraînement terminé pour tous les états.')

---
## Cellule 6 — Visualisation des courbes d'apprentissage

Affiche côte à côte les courbes Loss / OA / Kappa pour les deux états.

In [ ]:
from IPython.display import display, Image as IPImage

for state_slug in CFG.states:
    curves_png = RUNS_DIR / state_slug / 'training_curves.png'
    if curves_png.exists():
        print(f'\n── Courbes d\'apprentissage : {state_slug.title()} ──')
        display(IPImage(filename=str(curves_png)))
    else:
        print(f'Courbes non trouvées pour {state_slug}.')

---
## Cellule 7 — Matrices de confusion

In [ ]:
for state_slug in CFG.states:
    cm_png = RUNS_DIR / state_slug / 'test_confusion_matrix.png'
    if cm_png.exists():
        print(f'\n── Matrice de confusion (test) : {state_slug.title()} ──')
        display(IPImage(filename=str(cm_png)))
    else:
        print(f'Matrice non trouvée pour {state_slug}.')

---
## Cellule 8 — Tableau récapitulatif des métriques

In [ ]:
rows = []
for state_slug, summary in all_summaries.items():
    bv  = summary['best_val']
    tst = summary['test_at_best_val']
    rows.append({
        'État':           summary['state_name'],
        'Classes':        summary['num_classes'],
        'Val OA':         round(bv.get('oa',       0), 4),
        'Val F1':         round(bv.get('macro_f1', 0), 4),
        'Val Kappa':      round(bv.get('kappa',    0), 4),
        'Test OA':        round(tst.get('oa',       0), 4),
        'Test F1':        round(tst.get('macro_f1', 0), 4),
        'Test Kappa':     round(tst.get('kappa',    0), 4),
    })

df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))

# Sauvegarde CSV
summary_path = RUNS_DIR / 'summary_results.csv'
df_res.to_csv(summary_path, index=False, encoding='utf-8')
print(f'\nRésultats sauvegardés → {summary_path}')

---
## Cellule 9 — Ablation study (optionnel)

Reproduit les 4 variantes d'ablation du papier original.
Décommentez les lignes souhaitées et relancez.

In [ ]:
# Variantes disponibles :
#   no_alpe=True   -> MCTNet_noALPE  (PE sinusoïdale fixe)
#   no_mask=True   -> MCTNet_noMask  (ALPE sans masque de données manquantes)
#   no_cnn=True    -> MCTNet_noCnn   (branche CNN désactivée)
#   no_trans=True  -> MCTNet_noTrans (branche Transformer désactivée)

ABLATION_CONFIGS = [
    # ('no_alpe',  {'no_alpe':  True}),
    # ('no_mask',  {'no_mask':  True}),
    # ('no_cnn',   {'no_cnn':   True}),
    # ('no_trans', {'no_trans': True}),
]

ablation_results = {}

for config_name, overrides in ABLATION_CONFIGS:
    for state_slug in CFG.states:
        npz_path  = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.npz'
        json_path = PROCESSED_DIR / f'{state_slug}_mctnet_dataset.json'
        if not npz_path.exists(): continue

        bundle   = load_bundle(npz_path)
        metadata = json.loads(json_path.read_text(encoding='utf-8'))
        abl_run_dir = RUNS_DIR / f'{state_slug}_{config_name}'
        abl_run_dir.mkdir(parents=True, exist_ok=True)

        abl_args = SimpleNamespace(**vars(CFG))
        for k, v in overrides.items():
            setattr(abl_args, k, v)
        abl_args.output_dir      = str(abl_run_dir)
        abl_args.checkpoint_path = str(abl_run_dir / 'best_mctnet.pt')

        set_seed(CFG.seed)
        _, metrics = train_model(args=abl_args, bundle=bundle, metadata=metadata)

        key = f'{state_slug}_{config_name}'
        ablation_results[key] = {
            'state':  state_slug,
            'config': config_name,
            'test':   metrics['test_at_best_val'],
        }
        print(f'[{state_slug}|{config_name}] Test : {metrics["test_at_best_val"]}')

if not ABLATION_CONFIGS:
    print('Aucune ablation configurée — décommentez les lignes dans ABLATION_CONFIGS.')